# 05 — Évaluation Globale du Système

Ce notebook regroupe les métriques d'évaluation finales :
1. **ROUGE** (ROUGE-1, ROUGE-L) — Qualité des résumés
2. **BERTScore** — Similarité sémantique des réponses QA
3. **Exact Match / F1** — Précision du QA extractif
4. **Latence end-to-end** — Temps de réponse complet du pipeline RAG

In [ ]:
# Installation des dépendances d'évaluation
# !pip install rouge-score bert-score

In [ ]:
import sys, time
sys.path.insert(0, '..')

import numpy as np
from transformers import pipeline

## 1. ROUGE — Évaluation des Résumés

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

# Paires (résumé généré, résumé de référence)
summarizer = pipeline(
    "summarization",
    model="csebuetnlp/mT5_multilingual_XLSum",
    tokenizer="csebuetnlp/mT5_multilingual_XLSum"
)

eval_pairs = [
    {
        "source": "L'apprentissage automatique est un sous-domaine de l'intelligence artificielle qui permet aux ordinateurs d'apprendre à partir de données sans être explicitement programmés. Il existe trois catégories principales : l'apprentissage supervisé, non supervisé, et par renforcement.",
        "reference": "L'apprentissage automatique permet aux ordinateurs d'apprendre à partir de données. Il se divise en apprentissage supervisé, non supervisé et par renforcement."
    },
    {
        "source": "Les réseaux de neurones convolutifs (CNN) sont principalement utilisés pour le traitement d'images. Ils utilisent des filtres de convolution pour extraire des caractéristiques visuelles comme les contours, les textures et les formes. L'architecture typique comprend des couches de convolution, de pooling et des couches entièrement connectées.",
        "reference": "Les CNN utilisent des filtres de convolution pour extraire des caractéristiques d'images. Ils comprennent des couches de convolution, pooling et entièrement connectées."
    }
]

rouge_results = []
for pair in eval_pairs:
    generated = summarizer(pair["source"], max_length=150, min_length=20, num_beams=4)
    gen_text = generated[0]["summary_text"]
    
    scores = scorer.score(pair["reference"], gen_text)
    rouge_results.append(scores)
    
    print(f"\nSource  : {pair['source'][:80]}...")
    print(f"Généré  : {gen_text}")
    print(f"Réf.    : {pair['reference']}")
    print(f"ROUGE-1 : P={scores['rouge1'].precision:.3f} R={scores['rouge1'].recall:.3f} F={scores['rouge1'].fmeasure:.3f}")
    print(f"ROUGE-L : P={scores['rougeL'].precision:.3f} R={scores['rougeL'].recall:.3f} F={scores['rougeL'].fmeasure:.3f}")

# Moyenne
avg_r1_f = np.mean([r['rouge1'].fmeasure for r in rouge_results])
avg_rl_f = np.mean([r['rougeL'].fmeasure for r in rouge_results])
print(f"\n═══ Moyenne ROUGE ═══")
print(f"  ROUGE-1 F1 : {avg_r1_f:.3f}")
print(f"  ROUGE-L F1 : {avg_rl_f:.3f}")

## 2. BERTScore — Évaluation Sémantique QA

In [ ]:
from bert_score import score as bert_score

qa_pipeline = pipeline(
    "question-answering",
    model="illuin-technology/camembert-base-fquad",
    tokenizer="illuin-technology/camembert-base-fquad"
)

qa_eval = [
    {
        "question": "Qu'est-ce que l'apprentissage supervisé ?",
        "context": "L'apprentissage supervisé est une technique où le modèle apprend à partir de données étiquetées. Chaque exemple contient une entrée et la sortie attendue.",
        "reference": "une technique où le modèle apprend à partir de données étiquetées"
    },
    {
        "question": "Quel est le rôle de la rétropropagation ?",
        "context": "La rétropropagation du gradient permet d'ajuster les poids du réseau en propageant l'erreur de la sortie vers l'entrée. C'est l'algorithme fondamental pour l'entraînement des réseaux de neurones.",
        "reference": "ajuster les poids du réseau en propageant l'erreur"
    }
]

predictions = []
references = []

for qa in qa_eval:
    result = qa_pipeline(question=qa["question"], context=qa["context"])
    predictions.append(result["answer"])
    references.append(qa["reference"])
    
    print(f"\nQ: {qa['question']}")
    print(f"  Prédit    : {result['answer']}")
    print(f"  Référence : {qa['reference']}")
    print(f"  Confiance : {result['score']:.4f}")

# BERTScore
P, R, F1 = bert_score(predictions, references, lang="fr", verbose=False)
print(f"\n═══ BERTScore ═══")
for i, qa in enumerate(qa_eval):
    print(f"  Q{i+1}: P={P[i]:.3f} R={R[i]:.3f} F1={F1[i]:.3f}")
print(f"  Moyenne F1 : {F1.mean():.3f}")

## 3. Exact Match & Token F1

In [ ]:
import re
from collections import Counter

def normalize(text: str) -> str:
    """Normalisation pour comparaison."""
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', '', text)
    return ' '.join(text.split())

def exact_match(pred: str, ref: str) -> float:
    return float(normalize(pred) == normalize(ref))

def token_f1(pred: str, ref: str) -> float:
    pred_tokens = normalize(pred).split()
    ref_tokens = normalize(ref).split()
    common = Counter(pred_tokens) & Counter(ref_tokens)
    num_common = sum(common.values())
    if num_common == 0:
        return 0.0
    precision = num_common / len(pred_tokens)
    recall = num_common / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

em_scores = []
f1_scores = []

for pred, ref in zip(predictions, references):
    em = exact_match(pred, ref)
    f1 = token_f1(pred, ref)
    em_scores.append(em)
    f1_scores.append(f1)
    print(f"Prédit: '{pred}'")
    print(f"  Réf: '{ref}'")
    print(f"  EM={em:.0f} | Token-F1={f1:.3f}")

print(f"\n═══ Moyennes ═══")
print(f"  Exact Match : {np.mean(em_scores)*100:.1f}%")
print(f"  Token F1    : {np.mean(f1_scores):.3f}")

## 4. Latence End-to-End

In [ ]:
from backend.services.embedder import EmbeddingService

embedder = EmbeddingService()

# Simuler un pipeline RAG complet
context = qa_eval[0]["context"] * 3  # Simuler un contexte plus long
question = qa_eval[0]["question"]

N_RUNS = 5
timings = {"embedding": [], "qa": [], "total": []}

for _ in range(N_RUNS):
    t_start = time.perf_counter()
    
    # Étape 1: Embedding de la question
    t0 = time.perf_counter()
    q_emb = embedder.embed_query(question)
    timings["embedding"].append(time.perf_counter() - t0)
    
    # Étape 2: QA (FAISS search simulé — quasi-instantané)
    t0 = time.perf_counter()
    result = qa_pipeline(question=question, context=context)
    timings["qa"].append(time.perf_counter() - t0)
    
    timings["total"].append(time.perf_counter() - t_start)

print(f"═══ Latence End-to-End ({N_RUNS} runs) ═══\n")
for step, times in timings.items():
    print(f"  {step:12s} : {np.mean(times)*1000:7.1f} ms (moy) | {np.median(times)*1000:7.1f} ms (méd)")

print(f"\nNote : latence mesurée sur CPU. GPU réduit significativement le temps QA et embedding.")

## 5. Tableau récapitulatif

In [ ]:
print("╔═══════════════════════════════════════════╗")
print("║       ÉVALUATION FINALE — EduAI           ║")
print("╠═══════════════════════════════════════════╣")
print(f"║  ROUGE-1 F1        : {avg_r1_f:.3f}               ║")
print(f"║  ROUGE-L F1        : {avg_rl_f:.3f}               ║")
print(f"║  BERTScore F1      : {F1.mean():.3f}               ║")
print(f"║  Exact Match       : {np.mean(em_scores)*100:.1f}%               ║")
print(f"║  Token F1          : {np.mean(f1_scores):.3f}               ║")
print(f"║  Latence totale    : {np.mean(timings['total'])*1000:.0f} ms (moy CPU)    ║")
print("╠═══════════════════════════════════════════╣")
print("║  Tous modèles HuggingFace (gratuits)      ║")
print("║  Aucune API externe payante               ║")
print("╚═══════════════════════════════════════════╝")

## Conclusion

Le système EduAI montre des performances satisfaisantes :

- **Résumé (mT5)** : scores ROUGE acceptables, résumés cohérents en français
- **QA (CamemBERT)** : BERTScore élevé, réponses extraites pertinentes
- **Latence** : acceptable pour un usage interactif, surtout sur GPU
- **Zéro coût API** : tous les modèles sont gratuits (HuggingFace)

### Axes d'amélioration

- Augmenter le corpus d'évaluation (> 50 paires QA)
- Fine-tuner CamemBERT sur un dataset spécifique au domaine
- Utiliser un modèle de résumé abstractif plus récent si disponible
- Optimiser la latence avec ONNX Runtime ou quantification